In [30]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession


import $ivy.$
import org.apache.spark.sql.SparkSession

In [31]:
val spark = SparkSession.builder()
  .appName("MiPrimeraClaseSpark")
  .master("local[*]")
  .getOrCreate()

println(s"Sesión de Spark iniciada. Versión: ${spark.version}")

Sesión de Spark iniciada. Versión: 4.1.1


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@73ae0b43

In [32]:
val df = spark.createDataFrame(Seq(
  (1, "Scala"), 
  (2, "Spark"), 
  (3, "Jupyter")
)).toDF("id", "herramienta")

df.show()

+---+-----------+
| id|herramienta|
+---+-----------+
|  1|      Scala|
|  2|      Spark|
|  3|    Jupyter|
+---+-----------+



df: org.apache.spark.sql.package.DataFrame = [id: int, herramienta: string]

In [33]:
val miCVS = spark.read
  .option("header", "true")
  .option("sep", ";")          // Úsalo si tu CSV usa punto y coma en vez de coma
  .option("inferSchema", "true") // Para que detecte automáticamente si una columna es número o fecha
  .csv("C:\\Users\\Imp_06 - Mañana\\Downloads\\Detalle_sin_formato.csv")

// Para ver qué tipos de datos detectó (Integer, String, etc.)
miCVS.printSchema()

root
 |-- RUC / N? de Identificacion del Informado: integer (nullable = true)
 |-- Tipo de Registro: string (nullable = true)
 |-- Tipo de Comprobante: string (nullable = true)
 |-- Fecha de Emision: string (nullable = true)
 |-- Condicion de la Operacion: string (nullable = true)
 |-- Timbrado del Comprobante: integer (nullable = true)
 |-- Numero de Comprobante: string (nullable = true)
 |-- Monto Gravado 10%: double (nullable = true)
 |-- IVA 10%: double (nullable = true)
 |-- Monto Gravado 5%: double (nullable = true)
 |-- IVA 5%: integer (nullable = true)
 |-- Monto No Gravado / Exento : integer (nullable = true)
 |-- Total Comprobante: double (nullable = true)
 |-- Imputa IVA: string (nullable = true)
 |-- Imputa IRE: string (nullable = true)
 |-- Imputa IRP: string (nullable = true)
 |-- No Imputar: string (nullable = true)
 |-- Numero de Comprobante Asociado: string (nullable = true)
 |-- Timbrado del Comprobante Asociado: string (nullable = true)



miCVS: org.apache.spark.sql.package.DataFrame = [RUC / N? de Identificacion del Informado: int, Tipo de Registro: string ... 17 more fields]

In [34]:
import org.apache.spark.sql.functions._

val dfLimpio = miCVS

  // 1. Convertir columnas de dinero a Double por si acaso
  .withColumn("IVA 5%", col("IVA 5%").cast("double"))
  .withColumn("Monto No Gravado / Exento ", col("Monto No Gravado / Exento ").cast("double"))
  
  // 2. Convertir la Fecha de Emisión de texto a formato Fecha real
  // (Ajusta "dd/MM/yyyy" según cómo venga en tu CSV realmente)
  .withColumn("Fecha_Real", to_date(col("Fecha de Emision"), "dd/MM/yyyy"))

dfLimpio.printSchema()

root
 |-- RUC / N? de Identificacion del Informado: integer (nullable = true)
 |-- Tipo de Registro: string (nullable = true)
 |-- Tipo de Comprobante: string (nullable = true)
 |-- Fecha de Emision: string (nullable = true)
 |-- Condicion de la Operacion: string (nullable = true)
 |-- Timbrado del Comprobante: integer (nullable = true)
 |-- Numero de Comprobante: string (nullable = true)
 |-- Monto Gravado 10%: double (nullable = true)
 |-- IVA 10%: double (nullable = true)
 |-- Monto Gravado 5%: double (nullable = true)
 |-- IVA 5%: double (nullable = true)
 |-- Monto No Gravado / Exento : double (nullable = true)
 |-- Total Comprobante: double (nullable = true)
 |-- Imputa IVA: string (nullable = true)
 |-- Imputa IRE: string (nullable = true)
 |-- Imputa IRP: string (nullable = true)
 |-- No Imputar: string (nullable = true)
 |-- Numero de Comprobante Asociado: string (nullable = true)
 |-- Timbrado del Comprobante Asociado: string (nullable = true)
 |-- Fecha_Real: date (nullable 

import org.apache.spark.sql.functions._
dfLimpio: org.apache.spark.sql.package.DataFrame = [RUC / N? de Identificacion del Informado: int, Tipo de Registro: string ... 18 more fields]

In [36]:
val resumenIVA = dfLimpio.groupBy("Tipo de Comprobante")
  .agg(
    sum("IVA 10%").as("Total_IVA_10"),
    sum("IVA 5%").as("Total_IVA_5"),
    sum("Total Comprobante").as("Gran_Total")
  )

resumenIVA.show()

+-------------------+------------+-----------+----------+
|Tipo de Comprobante|Total_IVA_10|Total_IVA_5|Gran_Total|
+-------------------+------------+-----------+----------+
|            FACTURA|     148.841|      918.0|  1656.501|
+-------------------+------------+-----------+----------+



resumenIVA: org.apache.spark.sql.package.DataFrame = [Tipo de Comprobante: string, Total_IVA_10: double ... 2 more fields]